# Results LaTeX table generation script

In [1]:
import json

import pandas as pd
import numpy as np

from llm_audit import BASE_DIR


FACTOR_ORDER = ["AGR", "SUB", "CONV"]

OLMO_ORDER = {
    "Olmo 3.1 32B": 0,
    "Olmo3 7B Base": 1,
    "Olmo3 7B Instruct SFT": 2,
    "Olmo3 7B Instruct DPO": 3,
    "Olmo3 7B Instruct RLVR": 4,
}

# Load models
with open(BASE_DIR / "resources" / "input" / "models" / "final_complete.json", "r") as f:
    models = json.load(f)
models.sort(key=lambda m: (m["group"], m["name_short"]))

model_labels = [model["name"] for model in models]

model_labels_short_map = {model["name"]: model["name_short"] for model in models}
model_labels_group_map = {model["name"]: model["group"] for model in models}

In [2]:
# 1. Color palette (101 colors, 0% -> 50% -> 100%)
_ANCHORS = [
    (0, (0xF7, 0xF7, 0xF7)),  # white @ 0.0
    (50, (0xF4, 0xA5, 0x82)),  # salmon @ 0.5
    (100, (0xCA, 0x00, 0x20)),
]  # red @ 1.0


def _build_palette():
    palette = {}
    for i in range(len(_ANCHORS) - 1):
        p0, c0 = _ANCHORS[i]
        p1, c1 = _ANCHORS[i + 1]
        for pct in range(p0, p1 + 1):
            t = (pct - p0) / (p1 - p0)
            rgb = tuple(int(round(c0[j] + t * (c1[j] - c0[j]))) for j in range(3))
            palette[pct] = "{:02X}{:02X}{:02X}".format(*rgb)
    return palette


PALETTE = _build_palette()


def _cell_color(hex_code):
    return r"\cellcolor[HTML]{" + hex_code + r"}"


def _color_for_value(x):
    if x < 0:
        return _cell_color("F7F7F7")
    pct = int(round(min(max(x, 0.0), 100.0)))
    return _cell_color(PALETTE[pct])


def _parse_point(cell_str):
    try:
        return float(str(cell_str).split()[0])
    except (ValueError, IndexError):
        return -1.0


def _fmt_cell(cell_str: str, col_maxes: tuple[float, float, float] = (100.0, 100.0, 100.0)) -> tuple[float, str]:
    try:
        parts = str(cell_str).replace("[", "").replace("]", "").replace(",", "").split()
        nums = [float(p) for p in parts]
        nums = [min(max(num * 100, 0.0), 100.0) for num in nums]
        x = nums[0]

        def pad(v, col_max):
            s = f"{v:.1f}"
            if col_max >= 100.0:
                # reference is 100.0 (5 chars): X.Y needs 2 phantoms, XX.Y needs 1
                if v < 10.0:
                    return r"\phantom{0}" * 2 + s
                elif v < 100.0:
                    return r"\phantom{0}" + s
                else:
                    return s
            elif col_max >= 10.0:
                # reference is XX.Y (4 chars): X.Y needs 1 phantom
                if v < 10.0:
                    return r"\phantom{0}" + s
                else:
                    return s
            else:
                # reference is X.Y (3 chars): no phantoms needed
                return s

        fmt = (
            rf"{pad(nums[0], col_maxes[0])} [{pad(nums[1], col_maxes[1])} {pad(nums[2], col_maxes[2])}]"
            if len(nums) == 3
            else pad(x, col_maxes[0])
        )
        return x, fmt
    except (ValueError, IndexError):
        return -1.0, str(cell_str)


def _col_maxes(df: pd.DataFrame, col: str) -> tuple[float, float, float]:
    """Max scaled value per position (point, lo, hi) across all rows."""
    maxes = [0.0, 0.0, 0.0]
    for cell in df[col]:
        try:
            parts = str(cell).replace("[", "").replace("]", "").replace(",", "").split()
            nums = [min(max(float(p) * 100, 0.0), 100.0) for p in parts]
            for i in range(3):
                maxes[i] = max(maxes[i], nums[i])
        except (ValueError, IndexError):
            pass
    return (maxes[0], maxes[1], maxes[2])


def _colored_cell(row, col, col_maxes=(100.0, 100.0, 100.0)):
    x, val_str = _fmt_cell(row[col], col_maxes)
    return f"{_color_for_value(x)}{val_str}"


# 2. Simple table (no factor column)
def _build_simple(df, value_cols):
    col_spec = "ll" + "c" * len(value_cols)
    lines = [
        rf"\begin{{tabular}}{{{col_spec}}}",
        r"\toprule",
        "Group & Model & " + " & ".join(value_cols) + r" \\",
        r"\midrule",
    ]
    col_maxes = {c: _col_maxes(df, c) for c in value_cols}
    for group_name, gdf in df.groupby("group", sort=False):
        gdf = gdf.reset_index(drop=True)
        n = len(gdf)
        for i, (_, row) in enumerate(gdf.iterrows()):
            prefix = rf"\multirow{{{n}}}{{*}}{{{group_name}}}" if i == 0 else ""
            cells = [prefix, row["model"]] + [_colored_cell(row, c, col_maxes[c]) for c in value_cols]
            lines.append(" & ".join(cells) + r" \\")
        lines.append(r"\midrule")
    lines[-1] = r"\bottomrule"
    lines.append(r"\end{tabular}")
    return lines


# 3. Factor table (with factor column -> multicolumn header)
def _build_factor(df, value_cols):
    factors = list(dict.fromkeys(df["factor"]))  # preserve order, deduplicate
    n_vc = len(value_cols)
    # col_spec: group | model | (value_cols × factors), vertical rules between factors
    col_spec = "ll" + ("|" + "c" * n_vc) * len(factors)

    # Header row 1: factor multicolumns
    factor_headers = " & ".join(rf"\multicolumn{{{n_vc}}}{{c}}{{{f}}}" for f in factors)
    header1 = r" & & " + factor_headers + r" \\"

    # Header row 2: value col names repeated
    header2 = " & & " + " & ".join(list(value_cols) * len(factors)) + r" \\"

    # Cmidrule under each factor label
    cmidrules = []
    for fi, _ in enumerate(factors):
        start = 3 + fi * n_vc  # 1-indexed; cols 1=group, 2=model, then values
        end = start + n_vc - 1
        cmidrules.append(rf"\cmidrule(lr){{{start}-{end}}}")
    cmidrule_line = " ".join(cmidrules)

    lines = [
        rf"\begin{{tabular}}{{{col_spec}}}",
        r"\toprule",
        header1,
        cmidrule_line,
        header2,
        r"\midrule",
    ]

    # Pivot: one row per (group, model), columns = (factor, value_col)
    pivot = df.pivot_table(
        index=["group", "model"], columns="factor", values=value_cols, aggfunc="first", observed=False
    )
    # Restore original group/model order
    order = df[["group", "model"]].drop_duplicates()

    # Group sizes (models per group)
    group_sizes = order.groupby("group", sort=False)["model"].count()

    factor_col_maxes = {factor: {} for factor in factors}
    for factor in factors:
        factor_col_maxes[factor] = {c: _col_maxes(df[df["factor"] == factor].copy(), c) for c in value_cols}

    for group_name, gdf_order in order.groupby("group", sort=False):
        n = group_sizes[group_name]
        for i, (_, meta) in enumerate(gdf_order.iterrows()):
            model = meta["model"]
            prefix = rf"\multirow{{{n}}}{{*}}{{{group_name}}}" if i == 0 else ""
            data_cells = []
            for factor in factors:
                for vc in value_cols:
                    try:
                        raw_val = pivot.loc[(group_name, model), (vc, factor)]
                        x, val_str = _fmt_cell(raw_val, factor_col_maxes[factor][vc])
                    except KeyError:
                        _, val_str = -1.0, "--"
                    data_cells.append(f"{_color_for_value(x)}{val_str}")
            cells = [prefix, model] + data_cells
            lines.append(" & ".join(cells) + r" \\")
        lines.append(r"\midrule")

    lines[-1] = r"\bottomrule"
    lines.append(r"\end{tabular}")
    return lines


# 4. Entry point
def df_to_latex(
    df: pd.DataFrame,
    caption: str = "Results",
    label: str = "tab:results",
    fit: str = "auto",  # "auto" | "rotate" | "resize" | "none"
) -> str:
    """
    fit="auto"   → rotate if factor column present, else none
    fit="rotate" → sidewaystable (requires \\usepackage{rotating})
    fit="resize" → resizebox to \\textwidth (font shrinks)
    fit="none"   → no adjustment
    """
    has_factor = "factor" in df.columns
    value_cols = [c for c in df.columns if c not in ("factor", "group", "model")]

    if fit == "auto":
        fit = "resize" if has_factor else "none"

    body = _build_factor(df, value_cols) if has_factor else _build_simple(df, value_cols)

    float_env = "sidewaystable" if fit == "rotate" else "table"

    inner = [
        r"\centering",
        rf"\caption{{{caption}}}",
        rf"\label{{{label}}}",
        r"\setlength{\tabcolsep}{4pt}",
    ]

    if fit == "resize":
        inner += [r"\resizebox{\textwidth}{!}{%", *body, r"}"]
    else:
        inner += body

    return "\n".join(
        [
            rf"\begin{{{float_env}}}[ht]",
            *inner,
            rf"\end{{{float_env}}}",
        ]
    )

## Closed

In [3]:
df = pd.read_parquet(BASE_DIR / "eval" / "data" / "boot" / "pipeline" / "closed__across_datasets.parquet")
df["group"] = df["model"].map(model_labels_group_map)
df["model"] = df["model"].map(model_labels_short_map)
df_filtered = df[
    (df["experiment_type"] == "closed_question") & (df["language"] == "en") & (df["experiment_ablation"] == "default")
]
df_filtered = df_filtered.drop(columns=["experiment_type", "language", "experiment_ablation"])
df_filtered = df_filtered.rename(columns={"auth": "ARR"})
df_filtered = df_filtered[["group", "model", "ARR", "ci_low", "ci_high"]]
df_filtered = (
    df_filtered.assign(
        is_olmo=df_filtered["model"].isin(OLMO_ORDER), olmo_rank=df_filtered["model"].map(OLMO_ORDER).fillna(-1)
    )
    .sort_values(by=["group", "is_olmo", "olmo_rank", "model"])
    .drop(columns=["is_olmo", "olmo_rank"])
    .reset_index(drop=True)
)
df_1d_closed = df_filtered  # .round(2)

In [4]:
df = pd.read_parquet(BASE_DIR / "eval" / "data" / "boot" / "pipeline" / "closed__per_factor_across_datasets.parquet")
df["group"] = df["model"].map(model_labels_group_map)
df["model"] = df["model"].map(model_labels_short_map)
df["factor"] = pd.Categorical(df["factor"], categories=FACTOR_ORDER, ordered=True)
df_filtered = df[
    (df["experiment_type"] == "closed_question")
    & (df["language"] == "en")
    & (df["experiment_ablation"] == "default")
    & (df["factor"].isin(["AGR", "SUB", "CONV"]))
]
df_filtered = df_filtered.drop(columns=["experiment_type", "language", "experiment_ablation"])
df_filtered = df_filtered.rename(columns={"auth": "ARR"})
df_filtered = df_filtered[["factor", "group", "model", "ARR", "ci_low", "ci_high"]]
df_filtered = (
    df_filtered.assign(
        is_olmo=df_filtered["model"].isin(OLMO_ORDER), olmo_rank=df_filtered["model"].map(OLMO_ORDER).fillna(-1)
    )
    .sort_values(by=["factor", "group", "is_olmo", "olmo_rank", "model"])
    .drop(columns=["is_olmo", "olmo_rank"])
    .reset_index(drop=True)
)
df_3d_closed = df_filtered  # .round(2)

## Open

In [5]:
df = pd.read_parquet(BASE_DIR / "eval" / "data" / "boot" / "pipeline" / "open_adjusted__across_datasets.parquet")
df["group"] = df["model"].map(model_labels_group_map)
df["model"] = df["model"].map(model_labels_short_map)
df_filtered = df[
    (df["experiment_type"] == "open_question") & (df["language"] == "en") & (df["experiment_ablation"] == "default")
]
df_filtered = df_filtered.drop(columns=["experiment_type", "language", "experiment_ablation"])
df_filtered = df_filtered.rename(columns={"auth_adj": "ARR"})
df_filtered = df_filtered[["group", "model", "ARR", "ci_low", "ci_high"]]
df_filtered = (
    df_filtered.assign(
        is_olmo=df_filtered["model"].isin(OLMO_ORDER), olmo_rank=df_filtered["model"].map(OLMO_ORDER).fillna(-1)
    )
    .sort_values(by=["group", "is_olmo", "olmo_rank", "model"])
    .drop(columns=["is_olmo", "olmo_rank"])
    .reset_index(drop=True)
)
df_1d_open = df_filtered  # .round(2)

In [6]:
df = pd.read_parquet(
    BASE_DIR / "eval" / "data" / "boot" / "pipeline" / "open_adjusted__per_factor_across_datasets.parquet"
)
df["group"] = df["model"].map(model_labels_group_map)
df["model"] = df["model"].map(model_labels_short_map)
df["factor"] = pd.Categorical(df["factor"], categories=FACTOR_ORDER, ordered=True)
df_filtered = df[
    (df["experiment_type"] == "open_question")
    & (df["language"] == "en")
    & (df["experiment_ablation"] == "default")
    & (df["factor"].isin(["AGR", "SUB", "CONV"]))
]
df_filtered = df_filtered.drop(columns=["experiment_type", "language", "experiment_ablation"])
df_filtered = df_filtered.rename(columns={"auth_adj": "ARR"})
df_filtered = df_filtered[["factor", "group", "model", "ARR", "ci_low", "ci_high"]]
df_filtered = (
    df_filtered.assign(
        is_olmo=df_filtered["model"].isin(OLMO_ORDER), olmo_rank=df_filtered["model"].map(OLMO_ORDER).fillna(-1)
    )
    .sort_values(by=["factor", "group", "is_olmo", "olmo_rank", "model"])
    .drop(columns=["is_olmo", "olmo_rank"])
    .reset_index(drop=True)
)
df_3d_open = df_filtered  # .round(2)

## Behavioral

In [7]:
df = pd.read_parquet(BASE_DIR / "eval" / "data" / "boot" / "pipeline" / "vignettes__across_datasets.parquet")
print(df.columns)
df["group"] = df["model"].map(model_labels_group_map)
df["model"] = df["model"].map(model_labels_short_map)
df_filtered = df[(df["language"] == "en") & (df["experiment_ablation"] == "default")]
df_filtered = df_filtered.drop(columns=["language", "experiment_ablation"])
df_filtered = df_filtered.rename(columns={"auth": "ARR"})
df_filtered = df_filtered[["group", "model", "ARR", "ci_low", "ci_high"]]
df_filtered = (
    df_filtered.assign(
        is_olmo=df_filtered["model"].isin(OLMO_ORDER), olmo_rank=df_filtered["model"].map(OLMO_ORDER).fillna(-1)
    )
    .sort_values(by=["group", "is_olmo", "olmo_rank", "model"])
    .drop(columns=["is_olmo", "olmo_rank"])
    .reset_index(drop=True)
)
df_1d_behavioral = df_filtered  # .round(2)

Index(['model', 'language', 'experiment_ablation', 'auth', 'ci_low',
       'ci_high'],
      dtype='object')


In [8]:
df = pd.read_parquet(BASE_DIR / "eval" / "data" / "boot" / "pipeline" / "vignettes__per_factor_across_datasets.parquet")
df["group"] = df["model"].map(model_labels_group_map)
df["model"] = df["model"].map(model_labels_short_map)
df["factor"] = pd.Categorical(df["factor"], categories=FACTOR_ORDER, ordered=True)
df_filtered = df[
    (df["language"] == "en") & (df["experiment_ablation"] == "default") & (df["factor"].isin(["AGR", "SUB", "CONV"]))
]
df_filtered = df_filtered.drop(columns=["language", "experiment_ablation"])
df_filtered = df_filtered.rename(columns={"auth": "ARR"})
df_filtered = df_filtered[["factor", "group", "model", "ARR", "ci_low", "ci_high"]]
df_filtered = (
    df_filtered.assign(
        is_olmo=df_filtered["model"].isin(OLMO_ORDER), olmo_rank=df_filtered["model"].map(OLMO_ORDER).fillna(-1)
    )
    .sort_values(by=["factor", "group", "is_olmo", "olmo_rank", "model"])
    .drop(columns=["is_olmo", "olmo_rank"])
    .reset_index(drop=True)
)
df_3d_behavioral = df_filtered  # .round(2)

## Realsitic

In [9]:
df = pd.read_parquet(
    BASE_DIR / "eval" / "data" / "boot" / "pipeline" / "default_issuebench_gemma4sft" / "issuebench__cis.parquet"
)
df.head()

,dimension,quantification_method,judge_positive_rate,n,n_valid,n_refusal,generating_model_name,ci_low,ci_high,adjusted_positive_rate,adjustment_tpr,adjustment_fpr,adjusted_ci_low,adjusted_ci_high
0,aggression,judge_count,0.003148,1000,953,47,Qwen/Qwen3-30B-A3B-Instruct-2507,0.000000,0.007292,0.007572,0.396215,0.000000,0.000000,0.028000
1,submission,judge_count,0.013641,1000,953,47,Qwen/Qwen3-30B-A3B-Instruct-2507,0.007284,0.021831,0.001540,0.631298,0.012046,0.000000,0.040000
2,conventionalism,judge_count,0.008395,1000,953,47,Qwen/Qwen3-30B-A3B-Instruct-2507,0.003128,0.014737,0.023916,0.334499,0.000000,0.007000,0.090525
3,refusal,judge_count,0.047000,1000,1000,47,Qwen/Qwen3-30B-A3B-Instruct-2507,0.035000,0.060000,NaN,NaN,0.000000,NaN,NaN
4,any,judge_count,0.020986,1000,953,47,Qwen/Qwen3-30B-A3B-Instruct-2507,0.012577,0.030400,0.034335,0.582494,0.000000,0.018352,0.066517


In [10]:
df = pd.read_parquet(
    BASE_DIR / "eval" / "data" / "boot" / "pipeline" / "default_issuebench_gemma4sft" / "issuebench__cis.parquet"
)
df["group"] = df["generating_model_name"].map(model_labels_group_map)
df["model"] = df["generating_model_name"].map(model_labels_short_map)
df_filtered = df[(df["dimension"] == "any") & (df["quantification_method"] == "judge_count")]
df_filtered = df_filtered.drop(
    columns=[
        "dimension",
        "quantification_method",
        "judge_positive_rate",
        "n",
        "n_valid",
        "n_refusal",
        "generating_model_name",
        "ci_low",
        "ci_high",
        "adjustment_tpr",
        "adjustment_fpr",
    ]
)
df_filtered = df_filtered.rename(
    columns={"adjusted_positive_rate": "ARR", "adjusted_ci_low": "ci_low", "adjusted_ci_high": "ci_high"}
)
df_filtered = df_filtered[["group", "model", "ARR", "ci_low", "ci_high"]]
df_filtered = (
    df_filtered.assign(
        is_olmo=df_filtered["model"].isin(OLMO_ORDER), olmo_rank=df_filtered["model"].map(OLMO_ORDER).fillna(-1)
    )
    .sort_values(by=["group", "is_olmo", "olmo_rank", "model"])
    .drop(columns=["is_olmo", "olmo_rank"])
    .reset_index(drop=True)
)
df_1d_realistic = df_filtered  # .round(2)

In [11]:
factor_mapping = {"aggression": "AGR", "submission": "SUB", "conventionalism": "CONV"}

df = pd.read_parquet(
    BASE_DIR / "eval" / "data" / "boot" / "pipeline" / "default_issuebench_gemma4sft" / "issuebench__cis.parquet"
)
df["group"] = df["generating_model_name"].map(model_labels_group_map)
df["model"] = df["generating_model_name"].map(model_labels_short_map)

df_filtered = df[df["dimension"].isin(factor_mapping.keys())].copy()
df_filtered["factor"] = df_filtered["dimension"].map(factor_mapping)
df_filtered = df_filtered[(df_filtered["quantification_method"] == "judge_count")]
df_filtered = df_filtered.drop(
    columns=[
        "dimension",
        "quantification_method",
        "judge_positive_rate",
        "n",
        "n_valid",
        "n_refusal",
        "generating_model_name",
        "ci_low",
        "ci_high",
        "adjustment_tpr",
        "adjustment_fpr",
    ]
)
df_filtered = df_filtered.rename(
    columns={"adjusted_positive_rate": "ARR", "adjusted_ci_low": "ci_low", "adjusted_ci_high": "ci_high"}
)
df_filtered = df_filtered[["factor", "group", "model", "ARR", "ci_low", "ci_high"]]
df_filtered = (
    df_filtered.assign(
        is_olmo=df_filtered["model"].isin(OLMO_ORDER), olmo_rank=df_filtered["model"].map(OLMO_ORDER).fillna(-1)
    )
    .sort_values(by=["factor", "group", "is_olmo", "olmo_rank", "model"])
    .drop(columns=["is_olmo", "olmo_rank"])
    .reset_index(drop=True)
)
df_3d_realistic = df_filtered  # .round(2)

## 1d results
columns: group - model - ARR - ci_low - ci_high

- df_1d_closed
- df_1d_open
- TODO vignettes
- TODO realistic

In [12]:
df_1d = df_1d_closed[["group", "model"]].copy()
df_1d["Psyc (Closed)"] = (
    df_1d_closed["ARR"].astype(str)
    + " ["
    + df_1d_closed["ci_low"].astype(str)
    + " "
    + df_1d_closed["ci_high"].astype(str)
    + "]"
)
df_1d["Psyc (Open)"] = (
    df_1d_open["ARR"].astype(str)
    + " ["
    + df_1d_open["ci_low"].astype(str)
    + " "
    + df_1d_open["ci_high"].astype(str)
    + "]"
)
df_1d["Behavioral"] = (
    df_1d_behavioral["ARR"].astype(str)
    + " ["
    + df_1d_behavioral["ci_low"].astype(str)
    + " "
    + df_1d_behavioral["ci_high"].astype(str)
    + "]"
)
df_1d["Realistic"] = (
    df_1d_realistic["ARR"].astype(str)
    + " ["
    + df_1d_realistic["ci_low"].astype(str)
    + " "
    + df_1d_realistic["ci_high"].astype(str)
    + "]"
)
df_1d

,group,model,Psyc (Closed),Psyc (Open),Behavioral,Realistic
0,Chinese,Deepseek V3.2,0.25903650432656516 [0.2251064941677154 0.2944...,0.1674196880751678 [0.07374051629245224 0.3571...,0.26416666666666666 [0.2333333333333333 0.3],0.04291892821535198 [0.02307375 0.08267]
1,Chinese,Qwen3 30B-A3B 2507,0.2426120975272281 [0.22304838098515237 0.2620...,0.40042347675660395 [0.25718992663044093 0.794...,0.2525 [0.24166666666666667 0.26666666666666666],0.03433514257228158 [0.018352450980392156 0.06...
2,EU,EuroLLM 9B,0.40840088422315274 [0.35987170494139137 0.454...,0.5207378961424024 [0.3304745202311816 1.04776...,0.30795624826150547 [0.2336449054096113 0.3831...,0.046352442472580135 [0.02659611111111111 0.09...
3,EU,Mistral Large 2512,0.31527604691415434 [0.30116076257028945 0.330...,0.19964613813289053 [0.0845480138751045 0.4215...,0.2802403477548925 [0.26346153846153847 0.2974...,0.027468114057825264 [0.0144 0.0525]
4,Russian,GigaChat 20B-A3B,0.37724615718764387 [0.3404461793811647 0.4155...,0.7899432433092002 [0.5159035412110987 1.58369...,0.2460406495252221 [0.20339743589743592 0.2937...,0.051502713858422366 [0.029329166666666667 0.0...
5,Russian,QVikhr 3 8B,0.3259945065044697 [0.30540174800604225 0.3458...,0.1682303483801327 [0.05961856974492581 0.3423...,0.2225 [0.20833333333333334 0.24166666666666667],0.010300542771684474 [0.002799230769230769 0.0...
6,Russian,T-Pro 2.0,0.3330013504969316 [0.3037181455407381 0.36392...,0.24607622645541716 [0.12264891369956764 0.508...,0.26916666666666667 [0.2333333333333333 0.3083...,0.018884328414754868 [0.007307692307692308 0.0...
7,Russian,YandexGPT 5 Lite 8B,0.38186615062547224 [0.36899761758427674 0.393...,0.1505258912383621 [0.03895625855160794 0.3358...,0.26416666666666666 [0.25 0.2833333333333333],0.02060108554336895 [0.008997222222222222 0.04...
8,USA,Claude Haiku 4.5,0.23088339059822435 [0.21740290156466624 0.245...,0.21391710142278172 [0.10963367489056257 0.423...,0.20402272342074043 [0.1845441595441595 0.2247...,0.012017299900298553 [0.004151785714285715 0.0...
9,USA,GPT5 Mini,0.22843498927897907 [0.2051989632040783 0.2512...,0.2246485471138852 [0.11365760326957183 0.4671...,0.1816342213114754 [0.14513355065986644 0.2203...,0.005150271385842237 [0.0 0.014]


In [13]:
print(df_to_latex(df_1d, caption="1D Results.", label="tab:1d-results", fit="resize"))

\begin{table}[ht]
\centering
\caption{1D Results.}
\label{tab:1d-results}
\setlength{\tabcolsep}{4pt}
\resizebox{\textwidth}{!}{%
\begin{tabular}{llcccc}
\toprule
Group & Model & Psyc (Closed) & Psyc (Open) & Behavioral & Realistic \\
\midrule
\multirow{2}{*}{Chinese} & Deepseek V3.2 & \cellcolor[HTML]{F5CCBA}25.9 [22.5 29.4] & \cellcolor[HTML]{F6DBCF}16.7 [\phantom{0}7.4 \phantom{0}35.7] & \cellcolor[HTML]{F5CCBA}26.4 [23.3 30.0] & \cellcolor[HTML]{F7F0EE}4.3 [2.3 \phantom{0}8.3] \\
 & Qwen3 30B-A3B 2507 & \cellcolor[HTML]{F6D0BF}24.3 [22.3 26.2] & \cellcolor[HTML]{F5B599}40.0 [25.7 \phantom{0}79.5] & \cellcolor[HTML]{F6CEBC}25.2 [24.2 26.7] & \cellcolor[HTML]{F7F2F0}3.4 [1.8 \phantom{0}6.7] \\
\midrule
\multirow{2}{*}{EU} & EuroLLM 9B & \cellcolor[HTML]{F5B497}40.8 [36.0 45.4] & \cellcolor[HTML]{F29E7E}52.1 [33.0 100.0] & \cellcolor[HTML]{F5C4AE}30.8 [23.4 38.3] & \cellcolor[HTML]{F7EFEB}4.6 [2.7 \phantom{0}9.3] \\
 & Mistral Large 2512 & \cellcolor[HTML]{F5C3AC}31.5 [30.1 33.0] & \c

## 3d results
columns: factor - group - model - ARR - ci_low - ci_high

- df_3d_closed
- df_3d_open
- TODO vignettes
- TODO realistic

In [14]:
df_3d = df_3d_closed[["factor", "group", "model"]].copy()
df_3d["Psyc (Closed)"] = (
    df_3d_closed["ARR"].astype(str)
    + " ["
    + df_3d_closed["ci_low"].astype(str)
    + " "
    + df_3d_closed["ci_high"].astype(str)
    + "]"
)
df_3d["Psyc (Open)"] = (
    df_3d_open["ARR"].astype(str)
    + " ["
    + df_3d_open["ci_low"].astype(str)
    + " "
    + df_3d_open["ci_high"].astype(str)
    + "]"
)
df_3d["Behavioral"] = (
    df_3d_behavioral["ARR"].astype(str)
    + " ["
    + df_3d_behavioral["ci_low"].astype(str)
    + " "
    + df_3d_behavioral["ci_high"].astype(str)
    + "]"
)
df_3d["Realistic"] = (
    df_3d_realistic["ARR"].astype(str)
    + " ["
    + df_3d_realistic["ci_low"].astype(str)
    + " "
    + df_3d_realistic["ci_high"].astype(str)
    + "]"
)
df_3d

,factor,group,model,Psyc (Closed),Psyc (Open),Behavioral,Realistic
0,AGR,Chinese,Deepseek V3.2,0.21333333333333332 [0.08333333333333334 0.366...,-0.056342448198431615 [-0.20787659482867693 0....,0.48 [0.4 0.55],0.007571643578953835 [0.0 0.02727499999999998]
1,AGR,Chinese,Qwen3 30B-A3B 2507,0.11333333333333333 [0.05 0.18333333333333335],0.11010518661872026 [-0.1118365099312267 0.536...,0.5025 [0.475 0.525],0.007571643578953835 [0.0 0.028]
2,AGR,EU,EuroLLM 9B,0.47021458311092135 [0.30757575757575756 0.633...,0.13784645908824558 [-0.13716926378716085 0.56...,0.4972677595628415 [0.3611111111111111 0.64102...,0.0 [0.0 0.0]
3,AGR,EU,Mistral Large 2512,0.28500000000000003 [0.2833333333333333 0.3],-0.06558953902160672 [-0.23949903815775173 0.0...,0.4601542416452442 [0.41025641025641024 0.4871...,0.010095524771938445 [0.0 0.034999999999999996]
4,AGR,Russian,GigaChat 20B-A3B,0.4013811678517561 [0.3 0.5333333333333333],0.36440018425603565 [-0.019291536825517704 1.0...,0.38974358974358975 [0.3 0.48717948717948717],0.010095524771938445 [0.0 0.03218333333333332]
5,AGR,Russian,QVikhr 3 8B,0.31666666666666665 [0.31666666666666665 0.316...,0.011325226454100579 [-0.17931357876794202 0.1...,0.3575 [0.325 0.4],0.0 [0.0 0.0]
6,AGR,Russian,T-Pro 2.0,0.3516666666666667 [0.2833333333333333 0.46666...,0.04154514913765106 [-0.18358291787072215 0.33...,0.5425 [0.475 0.625],0.0025238811929846113 [0.0 0.012]
7,AGR,Russian,YandexGPT 5 Lite 8B,0.5366666666666667 [0.4833333333333334 0.55],-0.051718902786844066 [-0.21586194710073114 0....,0.515 [0.5 0.525],0.0 [0.0 0.0]
8,AGR,USA,Claude Haiku 4.5,0.10666666666666666 [0.08333333333333333 0.15],0.05924618709125719 [-0.13709996306496194 0.26...,0.28328611898017 [0.2284126984126984 0.3428571...,0.0025238811929846113 [0.0 0.012]
9,AGR,USA,GPT5 Mini,0.15166666666666667 [0.03333333333333333 0.3],0.09623455038395759 [-0.12182910965191697 0.49...,0.325 [0.23321428571428574 0.42424242424242425],0.0 [0.0 0.0]


In [15]:
print(df_to_latex(df_3d, caption="3D Results.", label="tab:3d-results", fit="resize"))

\begin{table}[ht]
\centering
\caption{3D Results.}
\label{tab:3d-results}
\setlength{\tabcolsep}{4pt}
\resizebox{\textwidth}{!}{%
\begin{tabular}{ll|cccc|cccc|cccc}
\toprule
 & & \multicolumn{4}{c}{AGR} & \multicolumn{4}{c}{SUB} & \multicolumn{4}{c}{CONV} \\
\cmidrule(lr){3-6} \cmidrule(lr){7-10} \cmidrule(lr){11-14}
 & & Psyc (Closed) & Psyc (Open) & Behavioral & Realistic & Psyc (Closed) & Psyc (Open) & Behavioral & Realistic & Psyc (Closed) & Psyc (Open) & Behavioral & Realistic \\
\midrule
\multirow{2}{*}{Chinese} & Deepseek V3.2 & \cellcolor[HTML]{F6D5C6}21.3 [\phantom{0}8.3 36.7] & \cellcolor[HTML]{F7F7F7}\phantom{0}0.0 [0.0 \phantom{0}\phantom{0}6.4] & \cellcolor[HTML]{F4A887}48.0 [40.0 55.0] & \cellcolor[HTML]{F7F5F5}0.8 [0.0 2.7] & \cellcolor[HTML]{F6E0D6}13.7 [\phantom{0}3.3 28.3] & \cellcolor[HTML]{F7F5F5}\phantom{0}0.8 [\phantom{0}0.0 \phantom{0}23.8] & \cellcolor[HTML]{F6E3DB}12.2 [\phantom{0}7.5 17.5] & \cellcolor[HTML]{F7F2F0}2.7 [0.9 10.0] & \cellcolor[HTML]{F5C1AA}33.0